**References**

  - [Generate fake data for Spring PetClinic with Pandas and Faker](https://www.feststelltaste.de/generate-fake-data-for-spring-petclinic-with-pandas-and-faker/)  
  - [Generating Fake CSV Data With Python](https://blog.dennisokeeffe.com/blog/2021-08-11-generating-fake-csv-data-with-python)  
  - [Generate (fake) data: Populating databases and DataFrames using simple Python packages](https://medium.com/@gilbertbarrios89/generate-fake-data-populating-databases-and-dataframes-using-simple-python-packages-that-can-1b17bc6c0144)  
  - [Faker documentation](https://faker.readthedocs.io/en/latest/)  
  - [Challenge: Two actors who cast together the most](https://www.codewars.com/kata/5818bde9559ff58bd90004a2/sql)

**Table of contents**<a id='toc0_'></a>    
- [Сonnecting to DB, creating tables](#toc1_)    
- [DB inspector](#toc2_)    
- [Data generating](#toc3_)    
  - [Data configuration](#toc3_1_)    
  - [actors table](#toc3_2_)    
  - [films table](#toc3_3_)    
  - [films_actors table](#toc3_4_)    
- [Saving data to DB](#toc4_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

In [1]:
# For data manipulations
import pandas as pd

# Work with DB
import os
from sqlalchemy import create_engine
from sqlalchemy import text
from sqlalchemy import URL
from sqlalchemy.inspection import inspect

# Data generation
import random
from datetime import datetime
from faker import Faker
from faker.providers import BaseProvider

# <a id='toc1_'></a>[Сonnecting to DB, creating tables](#toc0_)

Connect to movies DB with engine (for PostgreSQL you need to install the `psycopg2`).  
My parameters for connecting to DB are stored in a file `.env` (virtual environment).

In [2]:
url_object = URL.create('postgresql',
                        username=os.getenv('POSTGRES_USER'),
                        password=os.getenv('POSTGRES_PASSWORD'),
                        host=os.getenv('POSTGRES_HOST'),
                        port=os.getenv('POSTGRES_PORT'),
                        database=os.getenv('POSTGRES_DB'))

engine = create_engine(url_object)

Сreate tables in movies DB, data schema based on [this task](https://www.codewars.com/kata/5818bde9559ff58bd90004a2/sql).

In [3]:
query = """-- sql
CREATE TABLE films (
  film_id INTEGER NOT NULL,
  title CHARACTER VARYING(255) NOT NULL,
  genre CHARACTER VARYING(255),
  rating NUMERIC,
  release_date DATE,
  PRIMARY KEY (film_id)
);

CREATE TABLE actors (
  actor_id INTEGER NOT NULL,
  first_name CHARACTER VARYING(45) NOT NULL,
  last_name CHARACTER VARYING(45) NOT NULL,
  PRIMARY KEY (actor_id)
);

CREATE TABLE films_actors (
  actor_id INTEGER NOT NULL,
  film_id INTEGER NOT NULL,
  PRIMARY KEY (actor_id, film_id),
  FOREIGN KEY (actor_id) REFERENCES actors (actor_id),
  FOREIGN KEY (film_id) REFERENCES films (film_id)
);
"""
with engine.connect() as conn:
    conn.execute(text(query))
    conn.commit()

# <a id='toc2_'></a>[DB inspector](#toc0_)

Define an inspector of DB and check it's some useful features

In [4]:
insp = inspect(engine)

In [5]:
# Default schema
insp.default_schema_name

'public'

In [6]:
# Available get-methods for working with schema
get_methods = [x for x in dir(insp) if x.startswith('get')]
get_methods

['get_check_constraints',
 'get_columns',
 'get_domains',
 'get_enums',
 'get_foreign_keys',
 'get_foreign_table_names',
 'get_indexes',
 'get_materialized_view_names',
 'get_multi_check_constraints',
 'get_multi_columns',
 'get_multi_foreign_keys',
 'get_multi_indexes',
 'get_multi_pk_constraint',
 'get_multi_table_comment',
 'get_multi_table_options',
 'get_multi_unique_constraints',
 'get_pk_constraint',
 'get_schema_names',
 'get_sequence_names',
 'get_sorted_table_and_fkc_names',
 'get_table_comment',
 'get_table_names',
 'get_table_oid',
 'get_table_options',
 'get_temp_table_names',
 'get_temp_view_names',
 'get_unique_constraints',
 'get_view_definition',
 'get_view_names']

In [7]:
# Table names of movies DB
insp.get_table_names()

['actors', 'films_actors', 'films']

In [8]:
# Data types of actors table
pd.DataFrame(insp.get_columns('actors'))

,name,type,nullable,default,autoincrement,comment
0,actor_id,INTEGER,False,None,False,None
1,first_name,VARCHAR(45),False,None,False,None
2,last_name,VARCHAR(45),False,None,False,None


In [9]:
# PK of actors table
insp.get_pk_constraint('actors')

{'constrained_columns': ['actor_id'],
 'name': 'actors_pkey',
 'comment': None,
 'dialect_options': {'postgresql_include': []}}

In [10]:
# FK of films_actors table
insp.get_foreign_keys('films_actors')

[{'name': 'films_actors_actor_id_fkey',
  'constrained_columns': ['actor_id'],
  'referred_schema': None,
  'referred_table': 'actors',
  'referred_columns': ['actor_id'],
  'options': {},
  'comment': None},
 {'name': 'films_actors_film_id_fkey',
  'constrained_columns': ['film_id'],
  'referred_schema': None,
  'referred_table': 'films',
  'referred_columns': ['film_id'],
  'options': {},
  'comment': None}]

# <a id='toc3_'></a>[Data generating](#toc0_)

Get some real looking data using the fake data provider Faker.

In [11]:
# Initialize the Faker object
fake = Faker()

In [12]:
type(fake)

faker.proxy.Faker

In [13]:
# Example of random profile
fake.profile()

{'job': 'TEFL teacher',
 'company': 'Baker-Foster',
 'ssn': '270-65-6712',
 'residence': 'PSC 6780, Box 6976\nAPO AE 99396',
 'current_location': (Decimal('-32.3925135'), Decimal('78.145580')),
 'blood_group': 'O-',
 'website': ['http://www.lloyd.com/'],
 'username': 'aprilross',
 'name': 'Sheila Hernandez',
 'sex': 'F',
 'address': '47109 Alicia Spring\nWilsonbury, HI 61681',
 'mail': 'mendezmanuel@gmail.com',
 'birthdate': datetime.date(1988, 5, 2)}

## <a id='toc3_1_'></a>[Data configuration](#toc0_)

In [14]:
FILMS_COUNT = 100                 # 100 films
ACTORS_COUNT = 3 * FILMS_COUNT    # 300 actors

## <a id='toc3_2_'></a>[actors table](#toc0_)

In [15]:
# Structure of actors table
pd.DataFrame(insp.get_columns('actors'))

,name,type,nullable,default,autoincrement,comment
0,actor_id,INTEGER,False,None,False,None
1,first_name,VARCHAR(45),False,None,False,None
2,last_name,VARCHAR(45),False,None,False,None


In [16]:
# Filling of index (actor_id PK) from 1 to AMOUNT_ACTORS
actors = pd.DataFrame(index=range(1, ACTORS_COUNT + 1))
actors.index.name ='actor_id'

# Data generating for first_name and last_name
actors['first_name'] = actors.index.map(lambda x: fake.first_name())
actors['last_name'] = actors.index.map(lambda x: fake.last_name())

In [17]:
actors.head()

,first_name,last_name
actor_id,,
1,Michael,Andrade
2,Kimberly,Perez
3,Julie,Hernandez
4,Justin,Bates
5,Renee,Taylor


## <a id='toc3_3_'></a>[films table](#toc0_)

In [18]:
# Structure of films table
pd.DataFrame(insp.get_columns('films'))

,name,type,nullable,default,autoincrement,comment
0,film_id,INTEGER,False,None,False,None
1,title,VARCHAR(255),False,None,False,None
2,genre,VARCHAR(255),True,None,False,None
3,rating,NUMERIC,True,None,False,None
4,release_date,DATE,True,None,False,None


In [19]:
# Index filling (film_id PK)
films = pd.DataFrame(index=range(1, FILMS_COUNT + 1))
films.index.name = 'film_id'

You can create your own provider for some data:

In [20]:
# Create a new provider class for film.genre
class GenereProvider(BaseProvider):
    def film_genre(self):
        return random.choice(['Documentary', 'Thriller', 'Mystery',
                              'Horror', 'Action', 'Comedy', 'Drama', 'Romance'])

# Add new provider to Faker instance
fake.add_provider(GenereProvider)

In [21]:
# Get film.title
def get_film_title():
    words = fake.words()   # Returns 3 words
    capitalized_words = list(map(lambda x: x.capitalize(), words))
    return ' '.join(capitalized_words)


# Get film.rating [1.0, 10.0)
def get_film_rating():
    return round(random.uniform(1.0, 10.0), 1)


# Get film.date
def get_film_date():
    return fake.date_this_decade()

In [22]:
# Data generating for films table
films['title'] = films.index.map(lambda x: get_film_title())
films['genre'] = films.index.map(lambda x: fake.film_genre())
films['rating'] = films.index.map(lambda x: get_film_rating())
films['release_date'] = films.index.map(lambda x: get_film_date())

In [23]:
films.head()

,title,genre,rating,release_date
film_id,,,,
1,Again Teacher Continue,Romance,1.8,2026-07-21
2,Hit Prepare Likely,Horror,2.9,2026-02-16
3,Possible Care Scientist,Horror,8.3,2020-07-03
4,Blood Something Charge,Action,7.3,2020-06-04
5,Single Teacher Charge,Documentary,9.1,2022-09-06


## <a id='toc3_4_'></a>[films_actors table](#toc0_)

In [24]:
# Structure of films_actors table
pd.DataFrame(insp.get_columns('films_actors'))

,name,type,nullable,default,autoincrement,comment
0,actor_id,INTEGER,False,None,False,None
1,film_id,INTEGER,False,None,False,None


Get already existing entries (`actor_id` and `film_id`) from `actors` and `films` tables.  
Put initial films_actors len like `n x m = 30 000 rows`.

In [25]:
# Filling films_actors.film_id from films.film_id
# and films_actors.actor_id from actors.actor_id
films_actors = pd.DataFrame({'film_id': films.sample(len(actors) * len(films), replace=True).index,
                             'actor_id': actors.sample(len(actors) * len(films), replace=True).index})

films_actors = films_actors.drop_duplicates().set_index(['film_id', 'actor_id'])

films_actors

,
film_id,actor_id
12,237
91,164
70,264
48,5
47,95
...,...
19,75
73,50
37,52


# <a id='toc4_'></a>[Saving data to DB](#toc0_)

In [26]:
def store(df, table_name):
    df.to_sql(table_name, con=engine, if_exists='append', chunksize=100)

store(actors, 'actors')
store(films, 'films')
store(films_actors, 'films_actors')

In [27]:
engine.dispose()

Now you can use movies DB with fake data.  
For example, I found two actors who cast together the most and list titles of only those movies they were casting together:

```sql
WITH
  -- two actors who most frequently work together
  top_team AS (
    SELECT
      first.actor_id AS first_actor_id,
      second.actor_id AS second_actor_id,
      COUNT(*) AS cnt
    FROM films_actors AS first
    INNER JOIN films_actors AS second ON
      first.film_id = second.film_id     
      AND first.actor_id < second.actor_id
    GROUP BY
      first_actor_id,
      second_actor_id
    -- plain ORDER BY cnt DESC LIMIT 1 can be unstable on ties,
    -- use an explicit tie-breaker
    ORDER BY
      cnt DESC,
      first_actor_id,
      second_actor_id
    LIMIT 1
  ),
 
  -- films shared by the top acting pair
  top_team_films AS (
    SELECT
      fa1.film_id
    FROM films_actors AS fa1
    INNER JOIN films_actors AS fa2 ON
      fa1.film_id = fa2.film_id
      AND fa1.actor_id = (SELECT first_actor_id FROM top_team)
      AND fa2.actor_id = (SELECT second_actor_id FROM top_team)
  )

SELECT
  (a1.first_name || ' ' || a1.last_name) AS first_actor,
  (a2.first_name || ' ' || a2.last_name) AS second_actor,
  films.title
FROM top_team
INNER JOIN actors AS a1 ON top_team.first_actor_id = a1.actor_id
INNER JOIN actors AS a2 ON top_team.second_actor_id = a2.actor_id
INNER JOIN films ON films.film_id IN (SELECT film_id FROM top_team_films)
ORDER BY films.title ASC;

```